# How to calculate Elterngeld

Elterngeld (parental leave benefit) provides financial support to parents during the
first months after the birth of a child. It replaces a portion of the parent’s pre-birth
income — typically about 67% of net income. This net income is approximated using
simplified calculation rules. This document explains how to calculate Elterngeld based
on historical income data for a given household.

### Overview

Elterngeld calculation generally involves three main steps:

1. Calculate the parent’s average monthly gross income in the 12 months before the child’s birth.  
   Use this as input to compute  
   `("elterngeld", "mean_nettoeinkommen_für_bemessungsgrundlage_nach_geburt_m")`  
   applying the policy environment from the year **before** the child’s birth.

2. Calculate the parent’s gross income in the tax year prior to the child’s birth.  
   Use this to compute  
   `("einkommensteuer", "zu_versteuerndes_einkommen_y_sn")`  
   using the policy environment of the birth year.

3. Use the results from steps 1 and 2 as inputs to calculate the final Elterngeld amount 
   `("elterngeld", "betrag_m")` using the policy environment of the birth year.


## Example

Consider a child born in July 2024. The parent had a monthly gross wage of €2,000 in the
12 months before birth, and a yearly gross income of €26,000 in 2023.

An example Elterngeld application form from the state of Bavaria can be found [here](https://www.zbfs.bayern.de/imperia/md/content/zbfs_intranet/produktgruppe_i/formulare/beeg-antrag/2021_09_anlage_n_mit_info_.pdf).


### Step 1: Calculate Average Monthly Gross Income Before Birth

If income from dependent employment is the only source of income, the basis for the
Elterngeld calculation is the average monthly gross income in the 12 months before the
birth of a child. In our case, this would be July 2023 until June 2024. If other income
sources exist, the basis is the average monthly gross income in the calendar year before
the birth of the child, 2023 in our example. We will stick to the first case here
because it is the only thing that GETTSIM will handle directly right now.

> **Note**: The Elterngeld implementation in GETTSIM does currently not consider income
> from self-employment or other sources. See
> [this](https://github.com/ttsim-dev/gettsim/issues/613) issue.

So let's assume that income from dependent employment is the only source of income. If
you have only annual income data (and thus cannot compute gross income 12 months before
birth exactly), you must choose an approximation. In most cases, a good approximation is
the income of the year before birth.


In [ ]:
import numpy as np

from gettsim import InputData, MainTarget, TTTargets, main

The correct point in time for the data, i.e., the relevant state variables for
determining the net wage, are an interesting case. Many of those may change over time.
The relevant part is:

> Die entsprechenden Abzugsmerkmale werden den Lohn- oder Gehaltsabrechnungen entnommen.
> Grundsätzlich sind die Abzugsmerkmale maßgeblich, die in der überwiegenden Zahl der
> Monate des Bemessungszeitraums vor Geburt des Kindes gegolten haben.

So it is up to the data preparer to find out which tax class (Steuerklasse), church
tax (Kirchensteuer), health insurance, etc. to set. The Steuerklasse, for example, is always set
on a yearly basis; if it changes with the start of the calendar year a child is born,
the date of birth matters. Up to June, the Steuerklasse from the previous calendar
year will be relevant; for July or later, the Steuerklasse from the current calendar
year will be relevant.

GETTSIM provides an easy way to compute the relevant net wage
`("elterngeld", "mean_nettoeinkommen_in_12_monaten_vor_geburt_m")` using the target
`("elterngeld", "mean_nettoeinkommen_für_bemessungsgrundlage_nach_geburt_m")`.

We use the policy environment of January 1st of the year before the child was born (§2e
Abs. 1 S. 2 BEEG). Note that this is correct regardless of the point in time when the
child is born

In our case, this is January 1st, 2023.

Here, we use some example data with `("einnahmen", "bruttolohn_m")` being 2,000€ per
month.

In [ ]:
input_data_12_months_before_birth = {
    "alter": np.array([30]),
    "arbeitsstunden_w": np.array([39]),
    "familie": {
        "p_id_elternteil_1": np.array([-1]),
        "p_id_elternteil_2": np.array([-1]),
    },
    "kindergeld": {"in_ausbildung": np.array([False])},
    "lohnsteuer": {"steuerklasse": np.array([1])},
    "p_id": np.array([0]),
    "hh_id": np.array([0]),
    "sozialversicherung": {"pflege": {"beitrag": {"hat_kinder": np.array([False])}}},
    "wohnort_ost_hh": np.array([False]),
    "einnahmen": {
        "bruttolohn_m": np.array([2000]),
    },
}

Now, we calculate the approximated net wage in the 12 months before birth.

In [ ]:
net_wage_12_months_before_birth = main(
    main_target=MainTarget.results.tree,
    policy_date_str="2023-01-01",
    tt_targets=TTTargets.tree(
        {
            "elterngeld": {
                "mean_nettoeinkommen_für_bemessungsgrundlage_nach_geburt_m": None,
            }
        }
    ),
    input_data=InputData.tree(input_data_12_months_before_birth),
)["elterngeld"]["mean_nettoeinkommen_für_bemessungsgrundlage_nach_geburt_m"]
net_wage_12_months_before_birth

### Step 2: Taxable income in the tax year before birth

Parental leave benefits are paid out only if the parent's taxable income in the tax year
before the child was born is below a certain threshold. We calculate this using another
GETTSIM call.

First, you have to compile all the necessary data to compute the parent's taxable income
in the year before the child was born (here: 2023). Here, we just use some made-up data. 

In [ ]:
input_data_2023 = {
    "alter": np.array([30]),
    "arbeitsstunden_w": np.array([39]),
    "behinderungsgrad": np.array([0]),
    "einkommensteuer": {
        "abzüge": {
            "beitrag_private_rentenversicherung_m": np.array([0]),
            "kinderbetreuungskosten_m": np.array([0]),
            "p_id_kinderbetreuungskostenträger": np.array([0]),
        },
        "einkünfte": {
            "aus_forst_und_landwirtschaft": {"betrag_y": np.array([0])},
            "aus_gewerbebetrieb": {"betrag_y": np.array([0])},
            "aus_selbstständiger_arbeit": {"betrag_y": np.array([0])},
            "aus_vermietung_und_verpachtung": {"betrag_y": np.array([0])},
            "ist_hauptberuflich_selbstständig": np.array([False]),
            "sonstige": {
                "alle_weiteren_y": np.array([0]),
                "rente": {"betrag_m": np.array([0])},
            },
        },
        "gemeinsam_veranlagt": np.array([False]),
    },
    "einnahmen": {
        "bruttolohn_y": np.array([26000]),
        "kapitalerträge_y": np.array([0]),
    },
    "familie": {
        "alleinerziehend": np.array([False]),
        "p_id_ehepartner": np.array([-1]),
        "p_id_elternteil_1": np.array([-1]),
        "p_id_elternteil_2": np.array([-1]),
    },
    "geburtsjahr": np.array([1993]),
    "grundsicherung": {"im_alter": {"betrag_m_eg": np.array([0])}},
    "kindergeld": {
        "in_ausbildung": np.array([False]),
        "p_id_empfänger": np.array([-1]),
    },
    "p_id": np.array([1]),
    "sozialversicherung": {
        "kranken": {
            "beitrag": {
                "privat_versichert": np.array([False]),
                "bemessungsgrundlage_rente_m": np.array([0]),
            }
        },
        "pflege": {"beitrag": {"hat_kinder": np.array([False])}},
    },
    "wohnort_ost_hh": np.array([False]),
}

Now, we calculate the taxable income in 2023.

In [ ]:
taxable_income_2023 = main(
    main_target=MainTarget.results.tree,
    policy_date_str="2023-01-01",
    tt_targets=TTTargets.tree(
        {
            "einkommensteuer": {
                "zu_versteuerndes_einkommen_y_sn": None,
            }
        }
    ),
    input_data=InputData.tree(input_data_2023),
    include_warn_nodes=False,
)["einkommensteuer"]["zu_versteuerndes_einkommen_y_sn"]
taxable_income_2023

### Step 3: Compute Elterngeld

Finally, we can compute Elterngeld using the target `("elterngeld", "betrag_m")` and the
input data generated in step 1 and 2.

The reason for not doing this in one GETTSIM call (together with step 1 and 2) is that 
1. we have to use different policy environment dates for the two steps, and
2. other state variables of the household (e.g. place of residence) might change
after birth of the child.

First, we create the new data set after birth.

In [ ]:
input_data_after_birth = {
    "alter": np.array([31, 0]),
    "alter_monate": np.array([372, 0]),
    "arbeitsstunden_w": np.array([0, 0]),
    "bürgergeld": {"p_id_lebenspartner": np.array([-1, -1])},
    "einnahmen": {"bruttolohn_m": np.array([0, 0])},
    "elterngeld": {
        "bisherige_bezugsmonate": np.array([0, 0]),
        "claimed": np.array([True, False]),
        "mean_nettoeinkommen_in_12_monaten_vor_geburt_m": np.append(
            net_wage_12_months_before_birth, np.array([0])
        ),
        "zu_versteuerndes_einkommen_vorjahr_y_sn": np.append(
            taxable_income_2023, np.array([0])
        ),
    },
    "familie": {
        "alleinerziehend": np.array([True, False]),
        "p_id_elternteil_1": np.array([-1, 0]),
        "p_id_elternteil_2": np.array([-1, -1]),
    },
    "hh_id": np.array([0, 0]),
    "kindergeld": {"in_ausbildung": np.array([False, False])},
    "lohnsteuer": {"steuerklasse": np.array([1, 1])},
    "p_id": np.array([0, 1]),
    "sozialversicherung": {
        "pflege": {"beitrag": {"hat_kinder": np.array([True, False])}}
    },
    "wohnort_ost_hh": np.array([False, False]),
}

Then, we compute Elterngeld using the policy environment of the child's birthday.

In [ ]:
main(
    main_target=MainTarget.results.tree,
    policy_date_str="2024-07-01",
    tt_targets=TTTargets.tree(
        {
            "elterngeld": {
                "betrag_m": None,
            }
        }
    ),
    input_data=InputData.tree(input_data_after_birth),
)["elterngeld"]["betrag_m"]